# 실습 11. Cox 비례위험모형

사회약학 연구방법 노트의 '실습 11. Cox 비례위험모형'에 딸린 노트북입니다. R survival 패키지의 pbc 자료(Mayo Clinic 원발성 담즙성 담관염 무작위배정 시험 312명)로 사건과 결측을 정리하고, 단변수·다변수 Cox 모형을 적합하고, 비례위험 가정을 점검한 뒤 보정 생존곡선과 층화 Cox 모형을 만듭니다. 셀을 위에서부터 차례로 실행하세요. 설명과 출력 읽는 법은 사이트의 실습 11에 있습니다.


## a. 실습 데이터 준비

Colab에서는 런타임이 새로 배정될 때마다 lifelines를 설치합니다. 내 컴퓨터(Anaconda)에서는 한 번만 실행합니다.

### 셀 1. lifelines 설치

In [ ]:
!pip install lifelines

출력을 만든 패키지의 버전을 확인합니다. 내 화면의 출력이 사이트와 다를 때 먼저 비교합니다.

### 셀 2. 패키지 불러오기와 버전 확인

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib, statsmodels, lifelines

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("matplotlib  ", matplotlib.__version__)
print("statsmodels ", statsmodels.__version__)
print("lifelines   ", lifelines.__version__)

### 셀 3. PBC 자료 불러오기

In [ ]:
url = ("https://vincentarelbundock.github.io/Rdatasets/"
       "csv/survival/pbc.csv")
pbc = pd.read_csv(url)
print(pbc.shape)
pbc.head(3)

### 셀 4. 무작위배정 환자만 남기기

In [ ]:
print(pbc["trt"].value_counts(dropna=False))

d = pbc[pbc["trt"].notna()].copy()        # 무작위배정된 312명만
pd.crosstab(d["status"], d["trt"], margins=True)

### 셀 5. 사건, 시간, 치료 변수 만들기

In [ ]:
d["death"] = (d["status"] == 2).astype(int)   # 사망 = 1, 이식·생존 = 0
d["years"] = d["time"] / 365.25
d["dpca"] = (d["trt"] == 1).astype(int)       # D-penicillamine = 1
d["female"] = (d["sex"] == "f").astype(int)
d["age10"] = d["age"] / 10                    # 10세 단위

print("사망", d["death"].sum(), "명, 이식(중도절단)",
      (d["status"] == 1).sum(), "명")
d.groupby("dpca")["death"].agg(["count", "sum"])

### 셀 6. 결측값 확인과 분석용 자료 만들기

In [ ]:
na = d.isna().sum()
print(na[na > 0])
print("dropna() 전체 적용:", len(d.dropna()), "명")

cols = ["years", "death", "dpca", "age10", "female",
        "bili", "albumin", "edema", "protime"]
df = d[cols].dropna().reset_index(drop=True)
print("분석에 쓸 열만 dropna:", len(df), "명")

### 셀 7. 빌리루빈 로그 변환

In [ ]:
df["log2_bili"] = np.log2(df["bili"])      # 빌리루빈이 2배 -> 1 증가

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].hist(df["bili"], bins=30)
axes[0].set_xlabel("Bilirubin (mg/dL)")
axes[1].hist(df["log2_bili"], bins=30)
axes[1].set_xlabel("log2(bilirubin)")
axes[0].set_ylabel("Number of patients")
plt.tight_layout()
df[["bili", "log2_bili"]].describe().round(2)

## b. Cox 모형과 위험비

### 셀 8. 치료만 넣은 Cox 모형 (단변수)

In [ ]:
from lifelines import CoxPHFitter

cph1 = CoxPHFitter()
cph1.fit(df, duration_col="years", event_col="death", formula="dpca")
cph1.print_summary(decimals=3)

### 셀 9. 단변수 Cox 모형과 로그순위 검정 비교

In [ ]:
from lifelines.statistics import logrank_test

a = df["dpca"] == 1
lr = logrank_test(df.loc[a, "years"], df.loc[~a, "years"],
                  df.loc[a, "death"], df.loc[~a, "death"])
z = cph1.summary.loc["dpca", "z"]
print("log-rank chi2 :", round(lr.test_statistic, 4))
print("Wald z^2      :", round(z**2, 4))
print("LR chi2       :",
      round(cph1.log_likelihood_ratio_test().test_statistic, 4))

### 셀 10. 다변수 Cox 모형 (formula 사용)

In [ ]:
cph = CoxPHFitter()
cph.fit(df, duration_col="years", event_col="death",
        formula="dpca + age10 + female + log2_bili + albumin"
                " + C(edema) + protime")
cph.print_summary(decimals=3)

결측값이 있는 변수를 formula에 넣어 일부러 오류를 내 보는 셀(셀 11)은 이 노트북에서 뺐습니다(사이트의 실습 11 b 절에 있습니다).

### 셀 12. 위험비의 단위 바꾸기

In [ ]:
b = cph.params_                         # 계수(로그 위험비)
print("나이 1세당 HR   :", round(np.exp(b["age10"] / 10), 3))
print("나이 10세당 HR  :", round(np.exp(b["age10"]), 3))
print("나이 20세당 HR  :", round(np.exp(b["age10"] * 2), 3))
print("빌리루빈 2배 HR :", round(np.exp(b["log2_bili"]), 3))
print("빌리루빈 4배 HR :", round(np.exp(b["log2_bili"] * 2), 3))
print("알부민 0.5 g/dL 감소 HR:", round(np.exp(-0.5 * b["albumin"]), 3))

### 셀 13. 위험비와 누적위험의 비(상대위험도)는 다르다

In [ ]:
pts = pd.DataFrame([df.median()] * 2)       # 중앙값 환자 두 명
pts["dpca"] = 0
pts["edema"] = [0.0, 1.0]                   # 부종 없음 / 이뇨제에도 부종
S = cph.predict_survival_function(pts, times=[1, 5, 10])
risk = 1 - S                                # 누적 사망위험
risk.columns = ["no edema", "edema 1.0"]
risk["risk ratio"] = risk["edema 1.0"] / risk["no edema"]
print("HR (edema 1.0):", round(np.exp(b["C(edema)[T.1.0]"]), 2))
risk.round(3)

### 셀 14. 논문용 위험비 표 만들기

In [ ]:
tab = cph.summary[["exp(coef)", "exp(coef) lower 95%",
                   "exp(coef) upper 95%", "p"]].copy()
tab.columns = ["HR", "lower", "upper", "p"]
tab["HR (95% CI)"] = [f"{h:.2f} ({l:.2f}-{u:.2f})"
                      for h, l, u in zip(tab.HR, tab.lower, tab.upper)]
tab["P"] = [f"{p:.3f}" if p >= 0.001 else "<0.001" for p in tab.p]
tab[["HR (95% CI)", "P"]]

### 셀 15. 포레스트 그림

In [ ]:
labels = {"dpca": "D-penicillamine vs placebo",
          "age10": "Age (per 10 years)", "female": "Female vs male",
          "log2_bili": "Bilirubin (per doubling)",
          "albumin": "Albumin (per 1 g/dL)",
          "C(edema)[T.0.5]": "Edema 0.5 vs none",
          "C(edema)[T.1.0]": "Edema 1.0 vs none",
          "protime": "Prothrombin time (per 1 s)"}
fig, ax = plt.subplots(figsize=(7, 4))
cph.plot(hazard_ratios=True, ax=ax)
ax.set_xscale("log")
ax.set_xticks([0.25, 0.5, 1, 2, 4])
ax.set_xticklabels(["0.25", "0.5", "1", "2", "4"])
ax.minorticks_off()
names = [labels[t.get_text()] for t in ax.get_yticklabels()]
ax.set_yticklabels(names)
ax.set_xlabel("Hazard ratio (95% CI, log scale)")
plt.tight_layout()

### 셀 16. 빌리루빈을 로그로 넣은 이유 확인

In [ ]:
for name, f in [("bili", "bili"), ("log2_bili", "log2_bili")]:
    m = CoxPHFitter().fit(df, "years", "death",
        formula=f"dpca + age10 + female + {f} + albumin"
                " + C(edema) + protime")
    print(f"{name:<10} partial AIC = {m.AIC_partial_:.1f}"
          f"  C = {m.concordance_index_:.3f}")

## c. 비례위험 가정 점검

### 셀 17. 비례위험 검정 (척도화 Schoenfeld 잔차)

In [ ]:
from lifelines.statistics import proportional_hazard_test

ph = proportional_hazard_test(cph, df, time_transform="rank")
ph.print_summary(decimals=3)

### 셀 18. check_assumptions의 요약과 조언

In [ ]:
cph.check_assumptions(df, p_value_threshold=0.05)

### 셀 19. 척도화 Schoenfeld 잔차 그림

In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess

r = cph.compute_residuals(df, kind="scaled_schoenfeld")
t = df.loc[r.index, "years"]                 # 사망 시점
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for ax, v in zip(axes, ["dpca", "protime"]):
    beta_t = r[v] + cph.params_[v]           # 세로축 = beta(t)
    ax.scatter(t, beta_t, s=8, alpha=0.5)
    sm = lowess(beta_t, t, frac=0.5)         # 평활선
    ax.plot(sm[:, 0], sm[:, 1], color="black")
    ax.axhline(cph.params_[v], ls="--", color="gray")
    ax.set_xlabel("Years since randomization")
    ax.set_title(f"beta(t) for {v}")
axes[0].set_ylabel("Scaled Schoenfeld residual + coef")
plt.tight_layout()

### 셀 20. log(-log) 생존 그림 (부종 범주별)

In [ ]:
from lifelines import KaplanMeierFitter

fig, ax = plt.subplots(figsize=(6, 4))
for v, lab in [(0, "No edema"), (0.5, "Edema 0.5"), (1, "Edema 1.0")]:
    s = df[df["edema"] == v]
    k = KaplanMeierFitter().fit(s["years"], s["death"], label=lab)
    k.plot_loglogs(ax=ax)
ax.set_xlabel("log(years)")
ax.set_ylabel("log(-log S(t))")
plt.tight_layout()

### 셀 21. 추적 기간을 3년에서 나눈 긴 자료 만들기

In [ ]:
cut = 3                                     # 3년에서 나눔
x = df.assign(id=df.index)                  # 환자 번호 열
early = x.assign(start=0.0, stop=x["years"].clip(upper=cut), late=0)
early["event"] = np.where(x["years"] <= cut, x["death"], 0)
late = x[x["years"] > cut].assign(start=float(cut), late=1)
late["stop"] = late["years"]
late["event"] = late["death"]
long = pd.concat([early, late]).drop(columns=["years", "death"])
long["protime_late"] = long["protime"] * long["late"]
print(len(df), "명 ->", len(long), "줄")
show = ["id", "start", "stop", "event", "protime", "protime_late"]
long[long["id"].isin([0, 3])].sort_values(["id", "start"])[show]

### 셀 22. 기간별 위험비 (시간과의 교호작용)

In [ ]:
from lifelines import CoxTimeVaryingFitter

ctv = CoxTimeVaryingFitter()
ctv.fit(long, id_col="id", event_col="event",
        start_col="start", stop_col="stop",
        formula="dpca + age10 + female + log2_bili + albumin"
                " + C(edema) + protime + protime_late")
b2 = ctv.params_
print("protime HR, 0-3년 :", round(np.exp(b2["protime"]), 2))
print("protime HR, 3년 후:",
      round(np.exp(b2["protime"] + b2["protime_late"]), 2))
p_int = ctv.summary.loc["protime_late", "p"]
print("교호작용 P        :", round(p_int, 4))
print("dpca HR           :", round(np.exp(b2["dpca"]), 3))

## d. 보정 생존곡선과 층화 Cox

### 셀 23. 부종 범주별 예측 생존곡선 (나머지 변수는 중앙값)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
cph.plot_partial_effects_on_outcome("edema", [0, 0.5, 1], ax=ax)
ax.set_xlabel("Years since randomization")
ax.set_ylabel("Predicted survival probability")
plt.tight_layout()

### 셀 24. 직접 표준화한 보정 생존곡선

In [ ]:
grid = np.round(np.arange(0, 12.05, 0.1), 1)   # 0, 0.1, ..., 12년
surv = {}
for v, lab in [(1, "D-penicillamine"), (0, "Placebo")]:
    Si = cph.predict_survival_function(df.assign(dpca=v), times=grid)
    surv[lab] = Si.mean(axis=1)          # 312명 곡선의 평균
std = pd.DataFrame(surv)
print(std.loc[[1.0, 5.0, 10.0]].round(3).to_string())

fig, ax = plt.subplots(figsize=(6.5, 4))
std.plot(ax=ax, drawstyle="steps-post")
ax.set_xlabel("Years since randomization")
ax.set_ylabel("Adjusted survival probability")
ax.set_ylim(0, 1)
plt.tight_layout()

### 셀 25. 부종으로 층화한 Cox 모형

In [ ]:
cph_s = CoxPHFitter()
cph_s.fit(df, duration_col="years", event_col="death",
          formula="dpca + age10 + female + log2_bili + albumin"
                  " + protime",
          strata=["edema"])
cph_s.print_summary(decimals=3)

### 셀 26. 세 모형의 치료 효과 비교

In [ ]:
comp = pd.DataFrame({
    "unadjusted": cph1.summary.loc["dpca"],
    "adjusted": cph.summary.loc["dpca"],
    "stratified by edema": cph_s.summary.loc["dpca"]}).T
comp[["exp(coef)", "exp(coef) lower 95%",
      "exp(coef) upper 95%", "p"]].round(3)

## e. 과제 정답

**과제 1.** 셀 10의 다변수 모형에서 log2_bili 행을 Results 문장으로 옮기고, 이 숫자가 말해 주지 않는 것을 두 가지 이상 쓰세요. 빌리루빈 10에서 11 mg/dL로 오를 때의 위험비와, 셀 13의 방법으로 중앙값 환자의 빌리루빈이 두 배일 때 5년·10년 사망위험도 구해 보세요.

### 셀 27. 과제 1 정답. 빌리루빈의 위험비 읽기

In [ ]:
b_bili = cph.params_["log2_bili"]                # 두 배당 로그 위험비
for lo, hi in [(1, 2), (5, 10), (10, 11)]:
    hr = np.exp(b_bili * np.log2(hi / lo))       # lo에서 hi로 오를 때
    print(f"bilirubin {lo:>2} -> {hi:>2} mg/dL: HR {hr:.3f}")

pb = pd.DataFrame([df.median()] * 2)             # 중앙값 환자 두 명
pb["dpca"] = 0
pb["bili"] = [1.35, 2.7]                         # 빌리루빈 1.35와 그 두 배
pb["log2_bili"] = np.log2(pb["bili"])
rb = 1 - cph.predict_survival_function(pb, times=[5, 10])
rb.columns = ["bili 1.35", "bili 2.7"]
rb["risk ratio"] = rb["bili 2.7"] / rb["bili 1.35"]
rb.round(3)

**과제 2.** 간이식을 중도절단하지 않고 사건으로 세어(사망 또는 간이식) 셀 10의 다변수 모형을 다시 적합하세요. 사건 수, 치료의 위험비, 다른 변수의 위험비가 어떻게 바뀌는지 비교하고, 이식받은 환자가 어떤 환자였는지와 함께 설명하세요.

### 셀 28. 과제 2 정답. 사망 또는 간이식을 사건으로 보면

In [ ]:
df2 = d[cols + ["status"]].dropna().reset_index(drop=True)
df2["log2_bili"] = np.log2(df2["bili"])
df2["event2"] = (df2["status"] > 0).astype(int)   # 이식(1) 또는 사망(2)
print("events: death", df2["death"].sum(),
      " death or transplant", df2["event2"].sum())
cph_tx = CoxPHFitter()
cph_tx.fit(df2, duration_col="years", event_col="event2",
           formula="dpca + age10 + female + log2_bili + albumin"
                   " + C(edema) + protime")
s = cph_tx.summary.loc["dpca"]
print(f"dpca HR {s['exp(coef)']:.3f} ({s['exp(coef) lower 95%']:.3f}"
      f"-{s['exp(coef) upper 95%']:.3f}), p {s['p']:.3f}")
print(d.groupby(d["status"] == 1)[["age", "bili"]].median())  # 이식받은 19명(True)
pd.DataFrame({"death": cph.hazard_ratios_,
              "death or transplant": cph_tx.hazard_ratios_}).round(3)

**과제 3.** 동료가 치료 효과가 추적 기간에 따라 변하는지 보려고 `df["dpca_t"] = df["dpca"] * df["years"]` 열을 만들어 셀 10의 다변수 모형에 더했고, "치료와 시간의 교호작용이 뚜렷했다(위험비 연 0.68배, P < 0.001). 치료의 비례위험 가정이 깨졌다"라고 썼습니다. 무엇이 왜 틀렸는지 찾고, 추적 기간을 나눈 자료로 바르게 다시 적합해 보세요.

### 셀 29. 과제 3 정답. '치료 × 추적 시간' 열을 넣은 분석

In [ ]:
from lifelines.utils import to_episodic_format

f_t = ("dpca + age10 + female + log2_bili + albumin"
       " + C(edema) + protime + dpca_t")
wrong = df.assign(dpca_t=df["dpca"] * df["years"])   # 동료가 만든 열
cph_w = CoxPHFitter().fit(wrong, "years", "death", formula=f_t)

ep = to_episodic_format(df.assign(id=df.index), duration_col="years",
                        event_col="death", id_col="id", time_gaps=0.1)
ep["dpca_t"] = ep["dpca"] * ep["stop"]             # 그 구간 끝의 시간
ctv_t = CoxTimeVaryingFitter().fit(ep, id_col="id", event_col="death",
                                   start_col="start", stop_col="stop",
                                   formula=f_t)
print(len(df), "명 ->", len(ep), "줄")
print(f"P for dpca_t: wrong {cph_w.summary.loc['dpca_t', 'p']:.1e},"
      f" split {ctv_t.summary.loc['dpca_t', 'p']:.3f}")
pd.DataFrame({"wrong (years)": np.exp(cph_w.params_),
              "split (0.1 y)": np.exp(ctv_t.params_)}).round(3)